# structured-v15 dev-13

Launch this on Kaggle. Do not submit it to the leaderboard.

It runs the plain packed zip of `agents/structured-v15` on the 13-task dev cohort.
Three of those tasks are environment-unreachable (`fastapi_14246`, `fastapi_14262`,
`fastapi_14356`), so the reachable ceiling is 10. The notebook keeps traces and
prints resolved count, per-task wall time, no-edit runs, and calls to the first edit.

Nothing is registered. The hosted runner calls `compile_submission` with the host
tool registry, model registry, and code executor, and it does not pass a callback
registry. This notebook does not wrap that call. Unpack the zip and point the
official starter at that directory.


In [ ]:
import os
from pathlib import Path

TASK_IDS = [
    "fastapi_11194",
    "fastapi_14246",
    "fastapi_14262",
    "fastapi_14356",
    "fastapi_14786",
    "fastapi_14794",
    "rich_3105",
    "rich_3454",
    "rich_3472",
    "rich_3934",
    "rich_3938",
    "rich_3944",
    "rich_4070",
]
UNREACHABLE = {"fastapi_14246", "fastapi_14262", "fastapi_14356"}
REACHABLE_CEILING = len(TASK_IDS) - len(UNREACHABLE)
AGENT_DIR = Path(os.environ.get("AGENT_DIR", "/kaggle/working/agent"))
print("structured-v15", AGENT_DIR)
print("tasks", len(TASK_IDS), "reachable ceiling", REACHABLE_CEILING)
print("plain zip; no callback registry")

In [ ]:
def compile_plain(submission_dir, tool_registry, model_registry, code_executor):
    """Same arguments the hosted runner passes. No callback registry."""
    from adk_submission import compile_submission

    return compile_submission(
        submission_dir=submission_dir,
        tool_registry=tool_registry,
        model_registry=model_registry,
        code_executor=code_executor,
    )


print("compile_plain is the hosted call. The starter runs it on the unpacked zip.")

## After the official loop

Point `RESULTS` at the run directory that holds `task_results.jsonl` and
`results/traces/`. This cell does not start the model. Traces stay on disk.


In [ ]:
import json
import os
from pathlib import Path

UNREACHABLE = {"fastapi_14246", "fastapi_14262", "fastapi_14356"}
EDIT_MARKERS = ("line_fix", "cat >", "cat>>", "tee ")


def _events(trace):
    if isinstance(trace, dict):
        return trace.get("events") or trace.get("trace") or []
    return []


def _tool_name(event):
    if not isinstance(event, dict):
        return ""
    return str(event.get("tool") or event.get("name") or event.get("tool_name") or "")


def _is_edit(event):
    name = _tool_name(event)
    if name in {"edit_file", "write_file"}:
        return True
    blob = json.dumps(event, default=str)
    return any(marker in blob for marker in EDIT_MARKERS)


def summarize(rows, traces):
    tasks = []
    for row in rows:
        task_id = row.get("instance_id") or row.get("task_id") or ""
        trace = traces.get(task_id) or {}
        events = _events(trace)
        calls = 0
        first = None
        edited = False
        for event in events:
            if _tool_name(event):
                calls += 1
            if first is None and _is_edit(event):
                first = calls
                edited = True
        if not edited:
            patch = str(row.get("patch") or row.get("model_patch") or "")
            edited = bool(patch.strip())
        tasks.append(
            {
                "instance_id": task_id,
                "resolved": bool(row.get("resolved") or row.get("passed")),
                "wall_seconds": row.get("wall_seconds") or row.get("duration_seconds"),
                "no_edit": not edited,
                "calls_to_first_edit": first,
                "unreachable": task_id in UNREACHABLE,
            }
        )
    reachable = [item for item in tasks if not item["unreachable"]]
    return {
        "n": len(tasks),
        "resolved": sum(item["resolved"] for item in tasks),
        "reachable_ceiling": 10,
        "reachable_resolved": sum(item["resolved"] for item in reachable),
        "no_edit_runs": sum(item["no_edit"] for item in tasks),
        "tasks": tasks,
    }


RESULTS = Path(os.environ.get("V15_RESULTS", "/kaggle/working"))
rows = []
results_path = RESULTS / "task_results.jsonl"
if results_path.is_file():
    for line in results_path.read_text().splitlines():
        if line.strip():
            rows.append(json.loads(line))
traces = {}
trace_dir = RESULTS / "results" / "traces"
if not trace_dir.is_dir():
    trace_dir = RESULTS / "traces"
if trace_dir.is_dir():
    for path in sorted(trace_dir.glob("*.json")):
        payload = json.loads(path.read_text())
        task_id = payload.get("instance_id") or path.stem.removeprefix("trace_")
        traces[task_id] = payload
report = summarize(rows, traces)
print(
    "resolved",
    report["resolved"],
    "of",
    report["n"],
    "; reachable",
    report["reachable_resolved"],
    "of",
    report["reachable_ceiling"],
)
print("no_edit_runs", report["no_edit_runs"])
for item in report["tasks"]:
    print(
        item["instance_id"],
        "resolved" if item["resolved"] else "unresolved",
        "wall",
        item["wall_seconds"],
        "no_edit" if item["no_edit"] else "edit",
        "calls_to_first_edit",
        item["calls_to_first_edit"],
    )
out = RESULTS / "structured-v15-dev13-report.json"
out.write_text(json.dumps(report, indent=2) + "\n")
print("wrote", out)